# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [4]:
import os

In [5]:
# Write your code below.
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
import dask.dataframe as dd

c:\Users\israi\anaconda3\envs\dsi_participant\lib\site-packages\dask\dataframe\__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [17]:
import os
from glob import glob

# Write your code below.
price_data_dir = os.environ["PRICE_DATA"]
all_files = glob(os.path.join(price_data_dir, "*.csv"))


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [24]:
# Write your code below.
# If all_files is empty, you may need to update it to include parquet files, e.g.:
all_files = glob(os.path.join(price_data_dir, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(all_files).set_index("ticker")
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Adj_Close_lag_1 = x['Adj Close'].shift(1))
)
dd_feat = dd_feat.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)
dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(hi_lo_range = lambda x: x['High'] - x['Low'])
)
dd_feat

C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:5: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:8: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_feat.groupby('ticker', group_keys=False).apply(
C:\Users\israi\AppData\Local\Temp\ipykernel_28516\4241406197.py:14: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is une

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range
npartitions=90,,,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [25]:
# Write your code below.

pd_feat = dd_feat.compute()
pd_feat['ma_returns_10'] = pd_feat.groupby('ticker')['Returns'].rolling(10).mean().reset_index(level=0, drop=True)

pd_feat.head()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ma_returns_10
ticker,,,,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25,NaN
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46,NaN
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27,NaN
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30,NaN


In [27]:
pd_feat.sample(20)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,hi_lo_range,ma_returns_10
ticker,,,,,,,,,,,,,,
PPL,1982-02-17,0.000000,4.045769,3.987556,4.045769,0.033525,107300.0,PPL.csv,1982,4.016663,0.033284,0.007246,0.058213,0.002931
TEF,1999-08-09,13.020734,13.127461,13.002946,13.020734,2.553300,339600.0,TEF.csv,1999,12.807280,2.511443,0.016667,0.124516,-0.008058
PBI,1997-04-28,29.937500,30.562500,29.875000,30.500000,10.800749,471000.0,PBI.csv,1997,29.937500,10.601549,0.018789,0.687500,0.007043
GAZ,2012-07-03,3.210000,3.300000,3.180000,3.270000,3.270000,90700.0,GAZ.csv,2012,3.220000,3.220000,0.015528,0.120000,-0.004145
PLUG,2020-01-07,3.760000,3.900000,3.660000,3.810000,3.810000,22220600.0,PLUG.csv,2020,3.820000,3.820000,-0.002618,0.240000,0.027637
MRVL,2003-10-02,9.462500,9.690000,9.412500,9.515000,8.300258,5014400.0,MRVL.csv,2003,9.442500,8.237015,0.007678,0.277499,-0.006409
SPXC,2008-04-11,27.388567,27.605137,26.869806,27.129187,24.216537,2568000.0,SPXC.csv,2008,27.776379,24.794247,-0.023300,0.735331,0.002841
GLW,1994-06-09,10.833333,10.833333,10.708333,10.791667,4.950504,307800.0,GLW.csv,1994,10.833333,4.969616,-0.003846,0.125000,-0.004066
ALL,2002-01-18,32.040001,32.360001,31.700001,31.950001,20.498346,2497300.0,ALL.csv,2002,32.340000,20.748556,-0.012059,0.660000,-0.002215


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

 It is not necessary to convert to pandas to calculate the moving average return. Dask can perform the same operations. If the dataset is large then using Dask is better for such operations because it runs them in parallel. The rolling and groupby operations on Dask are slower than pandas so it is better to use pandas for small datasets.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.